In [44]:
"""
Preprocess and scale CICIDS2017 (source domain). Save scaler for reuse on target
dataset, and calculate covariance statistics.
"""

### Imports ###
import json
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [45]:
### Import and concatenate CSVs ###

# Creates a Path object pointing to the source-domain CSV directory.
data_dir = Path("data/raw/source")

# Read each CSV with encoding fallback for files that are not UTF-8.
def read_csv_with_fallback(file_path):
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(file_path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"Unable to decode {file_path}")

# Load all source CSV files into a list of DataFrames.
dfs = [read_csv_with_fallback(f) for f in data_dir.glob("*.csv")]

# Concatenate all DataFrames into one source-domain DataFrame.
df = pd.concat(dfs, ignore_index=True)

# Display a quick shape check and preview rows.
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (3119345, 85)


,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.5-104.16.207.165-54865-443-6,104.16.207.165,443.0,192.168.10.5,54865.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,192.168.10.5-104.16.28.216-55054-80-6,104.16.28.216,80.0,192.168.10.5,55054.0,6.0,7/7/2017 3:30,109.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,192.168.10.5-104.16.28.216-55055-80-6,104.16.28.216,80.0,192.168.10.5,55055.0,6.0,7/7/2017 3:30,52.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,192.168.10.16-104.17.241.25-46236-443-6,104.17.241.25,443.0,192.168.10.16,46236.0,6.0,7/7/2017 3:30,34.0,1.0,1.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,192.168.10.5-104.19.196.102-54863-443-6,104.19.196.102,443.0,192.168.10.5,54863.0,6.0,7/7/2017 3:30,3.0,2.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [46]:
### Data sanitization ###

# Handle missing values by replacing all occurrences of infinity with NaN
# then removing rows containing NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# remove duplicate flows and irrelevant columns
df.drop_duplicates(inplace=True)
df = df.drop(columns=["Flow ID", "Source IP", "Destination IP", "Timestamp"], errors="ignore")

# Remove leading/trailing spaces from all column names
df.rename(columns=lambda x: x.strip(), inplace=True)

In [47]:
### Feature-space alignment (select features according to predetermined shared feature space) ###

# Canonical feature-space contract shared by source and target pipelines.
FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")

# Handle missing file
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(
        f"Shared feature list not found at {FEATURE_LIST_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )

# Load canonical ordered features. Order must be preserved
with open(FEATURE_LIST_PATH, "r", encoding="utf-8") as f:
    shared_features = json.load(f)

# Identify label column and handle leading or trailing whitespace
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Aligns a DataFrame to the canonical shared feature contract by dropping extra
# columns, checking/optionally filling missing columns, and enforcing exact order.
# Used when source and target datasets must produce identical model input schema.
def align_feature_space(frame, feature_list, fill_missing=False, fill_value=0.0):
    
    # Compare incoming columns against the expected shared feature contract.
    feature_list = list(feature_list)
    incoming = set(frame.columns)
    expected = set(feature_list)

    extra = sorted(incoming - expected)
    missing = sorted(expected - incoming)

    # Strict mode (default): block pipeline if required features are absent.
    # This protects training/inference consistency across datasets.
    if missing and not fill_missing:
        preview = missing[:10]
        raise ValueError(
            f"Missing required features: {preview} (total={len(missing)})"
        )

    # Optional tolerant mode: create absent columns with a fixed value, then
    # continue with the canonical ordering.
    if missing and fill_missing:
        for col in missing:
            frame[col] = fill_value

    # Drop extras and enforce exact column order expected by downstream steps.
    aligned = frame[feature_list].copy()
    return aligned, extra, missing

# Align only feature columns; label handling happens separately.
feature_df = df.drop(columns=[label_col]).copy()
aligned_X, dropped_extra, missing_cols = align_feature_space(
    feature_df,
    shared_features,
    fill_missing=False,
    fill_value=0.0,
 )

# Reattach label column after enforcing the shared feature space contract.
df = pd.concat([aligned_X, df[[label_col]].reset_index(drop=True)], axis=1)

print(f"Loaded shared feature list from {FEATURE_LIST_PATH}")
print(f"Aligned feature count: {len(shared_features)}")
print(f"Dropped extra columns: {len(dropped_extra)}")
print(f"Missing required columns: {len(missing_cols)}")

Loaded shared feature list from data\processed\shared_feature_space.json
Aligned feature count: 68
Dropped extra columns: 15
Missing required columns: 0


In [48]:
### Label-space alignment (align labels according to predetermined shared label space) ###

SHARED_LABEL_SPACE_PATH = Path("data/processed/shared_label_space.json")
SOURCE_LABEL_MAP_PATH = Path("data/processed/source_label_map.json")

if not SHARED_LABEL_SPACE_PATH.exists():
    raise FileNotFoundError(
        f"Shared label space file not found at {SHARED_LABEL_SPACE_PATH}"
    )
if not SOURCE_LABEL_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Source label map file not found at {SOURCE_LABEL_MAP_PATH}"
    )

with open(SHARED_LABEL_SPACE_PATH, "r", encoding="utf-8") as f:
    shared_label_space = json.load(f)
with open(SOURCE_LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    source_label_map = json.load(f)

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Normalize raw labels for stable matching (trim spaces, keep missing as <NA>).
raw_labels = df[label_col].astype("string").str.strip()

# Detect genuinely missing labels (null/blank) separately from unmapped labels.
missing_label_mask = raw_labels.isna() | raw_labels.eq("")
missing_label_count = int(missing_label_mask.sum())

# Debug: unique raw labels before alignment (excluding missing values).
unique_before = sorted(raw_labels[~missing_label_mask].unique().tolist())
print(f"Unique labels before alignment: {len(unique_before)}")
print("Unique labels preview:", unique_before[:20])
print(f"Rows with missing label values: {missing_label_count}")

if missing_label_count > 0:
    missing_indices = df.index[missing_label_mask].tolist()
    preview_rows = missing_indices[:10]
    print(f"Missing-label row indices (preview): {preview_rows}")
    print("Raw label values at preview rows:", [repr(v) for v in raw_labels.loc[preview_rows].tolist()])
    raise ValueError(
        f"Missing label values found at row indices {preview_rows} "
        f"(total={missing_label_count}). Clean/drop these rows before label alignment."
    )

# Guardrail: mapping file must only map into allowed shared classes.
invalid_target_classes = sorted(
    set(source_label_map.values()) - set(shared_label_space)
)
if invalid_target_classes:
    raise ValueError(
        "source_label_map.json contains classes not present in shared_label_space.json: "
        f"{invalid_target_classes}"
    )

# Apply raw->shared mapping.
mapped_labels = raw_labels.map(source_label_map)

# Fail fast on unmapped non-missing raw labels to avoid silent label drift.
unmapped_mask = (~missing_label_mask) & mapped_labels.isna()
if unmapped_mask.any():
    unmapped_indices = df.index[unmapped_mask].tolist()
    unmapped_raw = raw_labels[unmapped_mask].tolist()
    preview_pairs = list(zip(unmapped_indices, unmapped_raw))[:10]
    print("Unmapped row index/raw label pairs (preview):", preview_pairs)

    unique_unmapped_raw = sorted(set(unmapped_raw))
    print("Unmapped raw labels (unique):", unique_unmapped_raw[:20])
    # repr() helps expose hidden whitespace or look-alike characters.
    print("Unmapped raw labels repr (unique):", [repr(v) for v in unique_unmapped_raw[:20]])

    raise ValueError(
        f"Unmapped raw labels found: {unique_unmapped_raw[:10]} "
        f"(total unique={len(unique_unmapped_raw)}, total rows={len(unmapped_indices)}). "
        f"See printed row indices/raw values above and update {SOURCE_LABEL_MAP_PATH}."
    )

# Replace dataset labels with aligned shared classes.
df[label_col] = mapped_labels

# Debug: unique labels after alignment.
unique_after = sorted(df[label_col].dropna().unique().tolist())
print(f"Unique labels after alignment: {len(unique_after)}/{len(shared_label_space)}")
print(unique_after)

Unique labels before alignment: 15
Unique labels preview: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack – Brute Force', 'Web Attack – Sql Injection', 'Web Attack – XSS']
Rows with missing label values: 291450
Missing-label row indices (preview): [2827677, 2827678, 2827679, 2827680, 2827681, 2827682, 2827683, 2827684, 2827685, 2827686]
Raw label values at preview rows: ['<NA>', '<NA>', '<NA>', '<NA>', '<NA>', '<NA>', '<NA>', '<NA>', '<NA>', '<NA>']


ValueError: Missing label values found at row indices [2827677, 2827678, 2827679, 2827680, 2827681, 2827682, 2827683, 2827684, 2827685, 2827686] (total=291450). Clean/drop these rows before label alignment.

In [ ]:
### Train/Val/Test Split ###
# todo

In [ ]:
### Scaling (save scaler for CIC_ToN_IoT) ###
# todo

In [ ]:
### Label encoding (save encoder for CIC_ToN_IoT) ###
# todo

In [ ]:
### Calculate and export covariance and mean statistics ###
# todo

In [ ]:
### Export processed data, final aligned feature list, and label mapping dictionary ###
# todo